# Hướng Dẫn Thực Nghiệm Phân Loại Ảnh X-Quang Phổi VinBigData

Notebook này hướng dẫn chi tiết từng bước trong quy trình Deep Learning chuẩn:
1. **Nạp & Tăng Cường Dữ Liệu (Data Augmentation & DataLoader)**
2. **Khởi tạo & So sánh 3 Kiến trúc Mô hình:**
   - `Model 1 - Simple CNN`: Viết từ đầu, tuân thủ nghiêm ngặt đan xen Conv2d và MaxPool2d.
   - `Model 2 - Complex CNN`: Viết từ đầu, kết hợp đường tuần tự (sequential) và đa nhánh song song (multi-path / parallel branches).
   - `Model 3 - Transfer Learning`: Sử dụng Pretrained Backbone (ResNet18/ResNet50).
3. **Huấn luyện Mô hình & Tự động lưu `best_model.pth`**
4. **Đánh giá trên Tập Test (Accuracy, Precision, Recall, F1 & Confusion Matrix)**

In [ ]:
# 1. Khởi tạo môi trường
import torch
import matplotlib.pyplot as plt
import numpy as np

import config
from data_loader import get_dataloaders
from models import get_model
from train import train
from eval import evaluate_model

print(f"PyTorch Version: {torch.__version__}")
print(f"Thiết bị tính toán: {config.DEVICE}")
print(f"Tổng số lớp nhãn bệnh lý: {config.NUM_CLASSES}")

## Bước 1: Khám Phá Dữ Liệu & Data Augmentation
Nạp DataLoader cho các tập `train`, `val`, `test`. Hệ thống tự động kích hoạt bộ dữ liệu demo nếu chưa có tập ảnh gốc.

In [ ]:
train_loader, val_loader, test_loader, class_names = get_dataloaders(batch_size=8)

# Trực quan hóa một batch ảnh mẫu
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for idx, ax in enumerate(axes):
    img_np = images[idx].permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = np.clip(img_np, 0, 1)
    ax.imshow(img_np)
    ax.set_title(f"{class_names[labels[idx]]}", fontsize=9)
    ax.axis("off")
plt.suptitle("Một số mẫu ảnh X-quang sau khi tiền xử lý và Data Augmentation", fontsize=12)
plt.show()

## Bước 2: Kiểm Tra Kiến Trúc 3 Mô Hình
Khởi tạo từng mô hình và kiểm tra kích thước Tensor đầu ra (`forward pass`).

In [ ]:
dummy_x = torch.randn(2, 3, 224, 224).to(config.DEVICE)

# Model 1: Simple CNN
m1 = get_model("simple", num_classes=15).to(config.DEVICE)
out1 = m1(dummy_x)
print(f"Model 1 (Simple CNN)       -> Output Shape: {out1.shape}")

# Model 2: Complex CNN
m2 = get_model("complex", num_classes=15).to(config.DEVICE)
out2 = m2(dummy_x)
print(f"Model 2 (Complex CNN)      -> Output Shape: {out2.shape}")

# Model 3: Transfer Learning
m3 = get_model("transfer", num_classes=15, pretrained=True).to(config.DEVICE)
out3 = m3(dummy_x)
print(f"Model 3 (Transfer ResNet)  -> Output Shape: {out3.shape}")

## Bước 3: Huấn Luyện Mô Hình & Tự Động Lưu `best_model.pth`
Tiến hành huấn luyện thử nghiệm mô hình trong một số epochs.

In [ ]:
# Chạy huấn luyện Model 1
history = train(model_name="simple", epochs=3, batch_size=16, learning_rate=1e-3)

# Trực quan hóa Loss & Accuracy qua các Epochs
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="Train Loss", marker="o")
ax1.plot(history["val_loss"], label="Val Loss", marker="s")
ax1.set_title("Hàm Mất Mát (Loss)")
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(True)

ax2.plot([a * 100 for a in history["train_acc"]], label="Train Acc", marker="o")
ax2.plot([a * 100 for a in history["val_acc"]], label="Val Acc", marker="s")
ax2.set_title("Độ Chính Xác (%)")
ax2.set_xlabel("Epoch")
ax2.legend()
ax2.grid(True)
plt.show()

## Bước 4: Đánh Giá Mô Hình Trên Tập Test
Nạp checkpoint tốt nhất `simple_best.pth` và đánh giá các chỉ số: Loss, Accuracy, Classification Report và vẽ Confusion Matrix.

In [ ]:
metrics = evaluate_model(model_name="simple", save_plot=True)
print(f"\nĐộ chính xác kiểm thử cuối cùng: {metrics['accuracy'] * 100:.2f}%")